In [3]:
# =============================================================================
# NOTEBOOK: 03_agent3_roi_computation.ipynb
# Automating Interview-Based Generative-AI ROI Measurement
#   — A Domain-Agnostic Three-Agent Pipeline (DSRM Design Artifact)
#
# AGENT 3 of 3 — ROI Computation & Sensitivity Analysis
#   Input : artifacts/inference/agent2_time.json   (per-node effort estimates)
#           artifacts/inference/process_graph.json (Metric B totals)
#           run_manifest.json                       (cost model)
#   Method: ReAct (Yao et al., 2023) interpreted for numeric work — the ONLY
#           actions that touch numbers are code actions:
#             (1) look up cost parameters, (2) compute S / ROI in Python,
#             (3) sweep 3 uncertain parameters for a sensitivity surface.
#           The LLM never does arithmetic (DP2); it only writes a one-paragraph
#           executive interpretation of the code-computed results.
#   Output: artifacts/inference/agent3_roi.json
#
# ROI formulas (paper Eq. 1 / Eq. 2), all evaluated in code:
#   S      = saved_hours_per_year * W                       (annual labor saving)
#   ROI(%) = (S - C_opex) / C_capex * 100
# where saved_hours_per_year is derived from Agent 2's AS-IS/TO-BE human effort.
#
# Sensitivity parameters (swept in code):
#   cases_per_month  (most uncertain — interview states no volume)
#   partial_retain   (TO-BE residual-human-effort assumption)
#   hourly_wage_usd  (region/role dependent)
#
# All example content and code are in English for journal submission.
# =============================================================================


# %%
# =============================================================================
# Cell 1. Bootstrap foundation from Notebook 00 (self-contained)
# =============================================================================
import os
import re
import json
import time
import hashlib
import itertools
import warnings
from pathlib import Path
from typing import Optional, Any

warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
from dotenv import load_dotenv
from openai import OpenAI

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent

DATA_RAW  = ROOT / "data" / "raw"
ARTIFACTS = ROOT / "artifacts"
INFER     = ARTIFACTS / "inference"
TAB       = ARTIFACTS / "tables"
CACHE     = ARTIFACTS / "llm_cache"
for p in (INFER, TAB, CACHE):
    p.mkdir(parents=True, exist_ok=True)


def rel(p) -> str:
    try:
        return str(Path(p).resolve().relative_to(ROOT.resolve()))
    except ValueError:
        return Path(p).name


SEED = 42
np.random.seed(SEED)

RUN_MANIFEST = ARTIFACTS / "run_manifest.json"
if not RUN_MANIFEST.exists():
    raise FileNotFoundError("[ERROR] run_manifest.json missing. Run nb 00 first.")
manifest = json.loads(RUN_MANIFEST.read_text(encoding="utf-8"))

MODELS       = manifest["models"]
DEFAULT_TIER = manifest["default_tier"]
MISSING      = manifest["missing_sentinel"]
COST_MODEL   = manifest["cost_model"]   # {hourly_wage_usd, capex_usd, opex_usd_per_year, ...}

load_dotenv(ROOT / ".env")
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
if not OPENAI_API_KEY:
    raise ValueError("[ERROR] OPENAI_API_KEY not set in .env.")
client = OpenAI(api_key=OPENAI_API_KEY)


# %%
# =============================================================================
# Cell 2. Re-declare cost tracker + llm_call() (identical to nb 00)
# =============================================================================
class CostTracker:
    def __init__(self):
        self.records: list[dict] = []

    def add(self, tier, model, usage, tag=""):
        p = MODELS.get(tier, {})
        pin, pcached, pout = p.get("in"), p.get("cached_in"), p.get("out")
        pt = getattr(usage, "prompt_tokens", 0) or 0
        ct = getattr(usage, "completion_tokens", 0) or 0
        cached = 0
        det = getattr(usage, "prompt_tokens_details", None)
        if det is not None:
            cached = getattr(det, "cached_tokens", 0) or 0
        fresh = max(pt - cached, 0)
        cost = None
        if None not in (pin, pout):
            pc = pcached if pcached is not None else pin
            cost = (fresh * pin + cached * pc + ct * pout) / 1_000_000
        self.records.append({"tag": tag, "tier": tier, "model": model,
                             "prompt_tokens": pt, "cached_tokens": cached,
                             "completion_tokens": ct, "cost_usd": cost})
        return cost or 0.0

    def total_usd(self):
        return float(sum(r["cost_usd"] or 0.0 for r in self.records))


COST = CostTracker()


def _safe_json(text):
    if not text:
        return None
    t = re.sub(r"^```(?:json)?\s*|\s*```$", "", text.strip(), flags=re.S).strip()
    try:
        return json.loads(t)
    except Exception:
        pass
    for opener, closer in (("{", "}"), ("[", "]")):
        i, j = t.find(opener), t.rfind(closer)
        if 0 <= i < j:
            try:
                return json.loads(t[i:j + 1])
            except Exception:
                continue
    return None


def _cache_key(model, system, user, temperature, response_json):
    raw = json.dumps({"m": model, "s": system, "u": user,
                      "t": temperature, "j": response_json},
                     ensure_ascii=False, sort_keys=True)
    return hashlib.sha256(raw.encode("utf-8")).hexdigest()[:24]


def llm_call(system, user, tier=DEFAULT_TIER, temperature=0.0,
             response_json=True, tag="", use_cache=True, max_retries=3):
    model = MODELS[tier]["name"]
    key = _cache_key(model, system, user, temperature, response_json)
    cache_file = CACHE / f"{key}.json"
    if use_cache and cache_file.exists():
        c = json.loads(cache_file.read_text(encoding="utf-8"))
        c["cached"] = True
        # Preserve the ORIGINAL (already-paid) cost so the ledger reflects the
        # true analysis cost even on cached re-runs; also re-log it to COST so
        # cumulative spend is complete regardless of cache state.
        original_cost = c.get("cost_usd", 0.0) or 0.0
        COST.records.append({
            "tag": tag or c.get("model", ""), "tier": c.get("tier", tier),
            "model": c.get("model", ""), "prompt_tokens": 0,
            "cached_tokens": 0, "completion_tokens": 0,
            "cost_usd": original_cost,
        })
        c["cost_usd"] = original_cost
        return c
    kwargs: dict[str, Any] = {
        "model": model,
        "messages": [{"role": "system", "content": system},
                     {"role": "user", "content": user}],
    }
    if response_json:
        kwargs["response_format"] = {"type": "json_object"}
    kwargs["temperature"] = temperature
    last_err = None
    for attempt in range(1, max_retries + 1):
        try:
            resp = client.chat.completions.create(**kwargs)
            text = resp.choices[0].message.content or ""
            parsed = _safe_json(text) if response_json else None
            if response_json and parsed is None:
                raise ValueError("Response was not valid JSON.")
            cost = COST.add(tier, model, resp.usage, tag=tag or model)
            out = {"text": text, "json": parsed, "cached": False,
                   "tier": tier, "model": model, "cost_usd": cost}
            if use_cache:
                cache_file.write_text(json.dumps(out, ensure_ascii=False),
                                      encoding="utf-8")
            return out
        except TypeError as e:
            if "temperature" in kwargs:
                kwargs.pop("temperature", None); last_err = e; continue
            last_err = e
        except Exception as e:
            last_err = e; time.sleep(min(2 ** attempt, 8))
    raise RuntimeError(f"[llm_call] failed after {max_retries} retries: {last_err}")


print("[INFO] CostTracker, llm_call() re-established for nb 03.")


# %%
# =============================================================================
# Cell 3. Load Agent-2 estimates + the cost model
# =============================================================================
A2_PATH = INFER / "agent2_time.json"
if not A2_PATH.exists():
    raise FileNotFoundError("[ERROR] agent2_time.json missing. Run nb 02 first.")
A2 = json.loads(A2_PATH.read_text(encoding="utf-8"))
estimates = A2["estimates"]
base_partial_retain = A2["params"]["partial_retain"]
default_cpm = A2["params"]["default_cases_per_month"]

W_BASE     = float(COST_MODEL["hourly_wage_usd"])
CAPEX      = float(COST_MODEL["capex_usd"])
OPEX_YEAR  = float(COST_MODEL["opex_usd_per_year"])

print(f"[INFO] Loaded {len(estimates)} node estimates from {rel(A2_PATH)}")
print(f"[INFO] Cost model: W=${W_BASE}/h  capex=${CAPEX:,.0f}  opex=${OPEX_YEAR:,.0f}/yr")
print(f"[INFO] Base partial_retain={base_partial_retain}, "
      f"default cases/month={default_cpm}")


# %%
# =============================================================================
# Cell 4. Core ROI engine — ALL arithmetic in code (DP2)
#
# We recompute AS-IS/TO-BE effort from per-node primitives so the engine can be
# re-evaluated under swept parameters. A node's monthly minutes scale linearly
# with cases_per_month, so we separate the per-case component from volume:
#
#   node_monthly_minutes(cpm) = minutes_per_case * cpm_scaled * qual_weight
# where cpm_scaled lets us override the (uncertain) volume globally in sweeps.
# =============================================================================
AI_LANES = {"system", "gpt", "ai", "ai agent"}

def is_ai(lane: str) -> bool:
    l = (lane or "").lower()
    return l in AI_LANES or "gpt" in l

def human_factor(grade: str, partial_retain: float) -> float:
    return {"full": 0.0, "partial": partial_retain, "manual": 1.0}.get(grade, 1.0)


def compute_effort(estimates: list[dict],
                   partial_retain: float,
                   cases_per_month_override: Optional[float] = None
                   ) -> tuple[float, float]:
    """Return (asis_human_minutes_per_month, tobe_human_minutes_per_month).

    If cases_per_month_override is given, EVERY node's volume is replaced by it
    (used by the sensitivity sweep, since the interview states no volumes). When
    None, each node keeps its own estimated cases_per_month.
    """
    asis_min = 0.0
    tobe_min = 0.0
    for e in estimates:
        cpm = cases_per_month_override if cases_per_month_override is not None \
            else float(e["cases_per_month"])
        monthly = float(e["minutes_per_case"]) * cpm * float(e.get("qual_weight", 1.0))
        if is_ai(e["lane"]):
            continue  # already AI in AS-IS -> contributes 0 human effort both sides
        asis_min += monthly
        tobe_min += monthly * human_factor(e["grade"], partial_retain)
    return asis_min, tobe_min


def compute_roi(asis_min: float, tobe_min: float,
                hourly_wage: float, capex: float, opex_year: float) -> dict:
    """Eq. 1 / Eq. 2, in code. Returns saving, ROI, payback, and reduction."""
    saved_min_per_month = max(asis_min - tobe_min, 0.0)
    saved_hours_per_year = saved_min_per_month * 12.0 / 60.0
    S = saved_hours_per_year * hourly_wage                     # Eq. 2 (annual saving)
    net_annual = S - opex_year
    roi_pct = (net_annual / capex * 100.0) if capex > 0 else float("nan")  # Eq. 1
    payback_years = (capex / net_annual) if net_annual > 0 else float("inf")
    reduction_pct = (1 - tobe_min / asis_min) * 100.0 if asis_min > 0 else 0.0
    return {
        "asis_min_month": asis_min,
        "tobe_min_month": tobe_min,
        "saved_hours_year": saved_hours_per_year,
        "annual_saving_usd": S,
        "net_annual_usd": net_annual,
        "roi_pct": roi_pct,
        "payback_years": payback_years,
        "effort_reduction_pct": reduction_pct,
    }


# --- Baseline ROI (each node keeps its own estimated volume) -----------------
asis0, tobe0 = compute_effort(estimates, base_partial_retain,
                              cases_per_month_override=None)
base = compute_roi(asis0, tobe0, W_BASE, CAPEX, OPEX_YEAR)

print("[INFO] --- Baseline ROI (code-computed) ---")
print(f"   AS-IS effort      : {base['asis_min_month']:,.0f} min/mo "
      f"({base['asis_min_month']/60:,.0f} h)")
print(f"   TO-BE effort      : {base['tobe_min_month']:,.0f} min/mo "
      f"({base['tobe_min_month']/60:,.0f} h)")
print(f"   Effort reduction  : {base['effort_reduction_pct']:.1f}%")
print(f"   Saved hours/year  : {base['saved_hours_year']:,.0f} h")
print(f"   Annual saving (S) : ${base['annual_saving_usd']:,.0f}")
print(f"   ROI               : {base['roi_pct']:,.0f}%")
print(f"   Payback           : {base['payback_years']:.2f} years")


# %%
# =============================================================================
# Cell 5. Sensitivity analysis — sweep the 3 uncertain parameters (code)
#
# The interview provides no volumes, so cases_per_month is the dominant
# uncertainty. We sweep it jointly with partial_retain and hourly_wage to give
# an honest ROI RANGE and a break-even surface, rather than a single number.
# =============================================================================
SWEEP = {
    "cases_per_month": [5, 10, 20, 40, 60, 100],   # dominant uncertainty
    "partial_retain":  [0.10, 0.30, 0.50],          # TO-BE assumption
    "hourly_wage_usd": [20, 35, 50, 80],            # region/role
}

rows = []
for cpm, pr, w in itertools.product(SWEEP["cases_per_month"],
                                    SWEEP["partial_retain"],
                                    SWEEP["hourly_wage_usd"]):
    a, t = compute_effort(estimates, pr, cases_per_month_override=cpm)
    r = compute_roi(a, t, w, CAPEX, OPEX_YEAR)
    rows.append({
        "cases_per_month": cpm, "partial_retain": pr, "hourly_wage_usd": w,
        "annual_saving_usd": round(r["annual_saving_usd"], 0),
        "roi_pct": round(r["roi_pct"], 1),
        "payback_years": round(r["payback_years"], 2)
        if np.isfinite(r["payback_years"]) else None,
        "effort_reduction_pct": round(r["effort_reduction_pct"], 1),
    })

sens_df = pd.DataFrame(rows)
print(f"[INFO] Sensitivity grid: {len(sens_df)} scenarios "
      f"({len(SWEEP['cases_per_month'])}×{len(SWEEP['partial_retain'])}"
      f"×{len(SWEEP['hourly_wage_usd'])}).")

# ROI range and break-even summary.
finite = sens_df.dropna(subset=["roi_pct"])
print(f"[INFO] ROI range across scenarios: "
      f"{finite['roi_pct'].min():,.0f}% to {finite['roi_pct'].max():,.0f}%")
be = sens_df[(sens_df["payback_years"].notna()) & (sens_df["payback_years"] <= 1.0)]
print(f"[INFO] Scenarios with <=1yr payback: {len(be)}/{len(sens_df)}")

# Break-even volume at the base wage & partial_retain: smallest cpm with ROI>0.
be_base = sens_df[(sens_df["hourly_wage_usd"] == W_BASE) &
                  (sens_df["partial_retain"] == base_partial_retain) &
                  (sens_df["roi_pct"] > 0)].sort_values("cases_per_month")
if not be_base.empty:
    print(f"[INFO] Break-even volume @ base params: "
          f"cases/month >= {be_base.iloc[0]['cases_per_month']:.0f}")


# %%
# =============================================================================
# Cell 6. ReAct interpretation — LLM writes the executive summary ONLY
#
# The model receives the code-computed numbers and produces a concise, honest
# managerial paragraph. It is explicitly forbidden from introducing new numbers
# or recomputing anything (DP2). This is the 'reasoning' half of ReAct; the
# 'action' half (lookup + compute) was done in code above.
# =============================================================================
summary_payload = {
    "baseline": {
        "effort_reduction_pct": round(base["effort_reduction_pct"], 1),
        "saved_hours_year": round(base["saved_hours_year"], 0),
        "annual_saving_usd": round(base["annual_saving_usd"], 0),
        "roi_pct": round(base["roi_pct"], 0),
        "payback_years": round(base["payback_years"], 2),
    },
    "sensitivity": {
        "roi_min_pct": round(float(finite["roi_pct"].min()), 0),
        "roi_max_pct": round(float(finite["roi_pct"].max()), 0),
        "break_even_cases_per_month":
            int(be_base.iloc[0]["cases_per_month"]) if not be_base.empty else None,
    },
    "cost_model": {"capex_usd": CAPEX, "opex_usd_per_year": OPEX_YEAR,
                   "hourly_wage_usd": W_BASE},
    "caveat": "All figures derive from prior/implied time estimates; the "
              "interview stated no explicit durations or volumes.",
}

REACT_SYSTEM = """\
You are the reporting layer of an ROI-analysis pipeline. You are given
ALREADY-COMPUTED figures (a baseline scenario and a sensitivity range). Write a
single, sober executive paragraph (4-6 sentences) that a manager could read.

Strict rules:
  - Use ONLY the numbers provided. Do NOT invent, recompute, or round away any
    figure. Do NOT add numbers that are not in the input.
  - State the baseline ROI and payback, then the sensitivity range, then the
    key caveat about estimate provenance honestly.
  - No hype. Present the result as conditional on the stated assumptions.

Return ONLY JSON: {"executive_summary": "<one paragraph>"}
"""

REACT_USER = "Computed figures:\n" + json.dumps(summary_payload, ensure_ascii=False, indent=1)

react = llm_call(system=REACT_SYSTEM, user=REACT_USER,
                 tier=DEFAULT_TIER, temperature=0.0,
                 response_json=True, tag="agent3_react_summary")
exec_summary = (react["json"] or {}).get("executive_summary", "").strip()
print("[INFO] Executive summary (LLM, interpretation only):")
print(exec_summary)


# %%
# =============================================================================
# Cell 7. Persist Agent-3 output + paper tables
# =============================================================================
OUT_PATH = INFER / "agent3_roi.json"
out_obj = {
    "agent": "agent3_roi_computation",
    "cost_model": {"hourly_wage_usd": W_BASE, "capex_usd": CAPEX,
                   "opex_usd_per_year": OPEX_YEAR},
    "baseline": base,
    "sensitivity_sweep": SWEEP,
    "sensitivity_grid": rows,
    "executive_summary": exec_summary,
    "provenance_caveat": summary_payload["caveat"],
}
# Convert any inf payback to a JSON-safe marker.
def _json_safe(o):
    if isinstance(o, float) and not np.isfinite(o):
        return None
    if isinstance(o, dict):
        return {k: _json_safe(v) for k, v in o.items()}
    if isinstance(o, list):
        return [_json_safe(v) for v in o]
    return o

OUT_PATH.write_text(json.dumps(_json_safe(out_obj), ensure_ascii=False, indent=2),
                    encoding="utf-8")
print(f"[INFO] Agent-3 output -> {rel(OUT_PATH)}")

# Sensitivity table for the paper appendix.
sens_csv = TAB / "agent3_sensitivity.csv"
sens_df.to_csv(sens_csv, index=False, encoding="utf-8-sig")
print(f"[INFO] Sensitivity table -> {rel(sens_csv)}")

# A compact baseline table too.
base_row = pd.DataFrame([{
    "effort_reduction_pct": round(base["effort_reduction_pct"], 1),
    "saved_hours_year": round(base["saved_hours_year"], 0),
    "annual_saving_usd": round(base["annual_saving_usd"], 0),
    "roi_pct": round(base["roi_pct"], 0),
    "payback_years": round(base["payback_years"], 2),
}])
base_csv = TAB / "agent3_baseline_roi.csv"
base_row.to_csv(base_csv, index=False, encoding="utf-8-sig")
print(f"[INFO] Baseline ROI table -> {rel(base_csv)}")
print(f"[INFO] Agent-3 spend this run: ${COST.total_usd():.5f}")

[INFO] CostTracker, llm_call() re-established for nb 03.
[INFO] Loaded 46 node estimates from artifacts\inference\agent2_time.json
[INFO] Cost model: W=$35.0/h  capex=$60,000  opex=$12,000/yr
[INFO] Base partial_retain=0.3, default cases/month=20.0
[INFO] --- Baseline ROI (code-computed) ---
   AS-IS effort      : 41,050 min/mo (684 h)
   TO-BE effort      : 19,755 min/mo (329 h)
   Effort reduction  : 51.9%
   Saved hours/year  : 4,259 h
   Annual saving (S) : $149,065
   ROI               : 228%
   Payback           : 0.44 years
[INFO] Sensitivity grid: 72 scenarios (6×3×4).
[INFO] ROI range across scenarios: -14% to 780%
[INFO] Scenarios with <=1yr payback: 31/72
[INFO] Break-even volume @ base params: cases/month >= 10
[INFO] Executive summary (LLM, interpretation only):
The baseline ROI is 228.0% with a payback period of 0.44 years, based on an effort reduction of 51.9% and annual savings of $149,065 from 4,259 saved hours. The sensitivity analysis indicates a potential ROI range 